In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (6362620, 11)


In [3]:
print(df.isnull().sum())

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [4]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [5]:
financial_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

for column in financial_columns:
    negative_count = (df[column] < 0).sum()
    print(f"{column}: {negative_count} negative values")

amount: 0 negative values
oldbalanceOrg: 0 negative values
newbalanceOrig: 0 negative values
oldbalanceDest: 0 negative values
newbalanceDest: 0 negative values


In [6]:
zero_amount_count = (df["amount"] == 0).sum()

print("Zero-amount transactions:", zero_amount_count)

Zero-amount transactions: 16


In [7]:
print(df["isFraud"].value_counts())
print("\nUnique target values:", df["isFraud"].unique())

isFraud
0    6354407
1       8213
Name: count, dtype: int64

Unique target values: [0 1]


In [8]:
print(df["type"].unique())

<StringArray>
['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN']
Length: 5, dtype: str


In [9]:
expected_types = {
    "PAYMENT",
    "TRANSFER",
    "CASH_OUT",
    "DEBIT",
    "CASH_IN"
}

actual_types = set(df["type"].unique())

print("Unexpected transaction types:",
      actual_types - expected_types)

Unexpected transaction types: set()


In [10]:
print("nameOrig unique values:", df["nameOrig"].nunique())
print("nameDest unique values:", df["nameDest"].nunique())

print("Total rows:", len(df))

nameOrig unique values: 6353307
nameDest unique values: 2722362
Total rows: 6362620


In [11]:
print(
    pd.crosstab(
        df["isFlaggedFraud"],
        df["isFraud"]
    )
)

isFraud               0     1
isFlaggedFraud               
0               6354407  8197
1                     0    16


In [12]:
origin_balance_difference = (
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
)

print(origin_balance_difference.describe())

count    6.362620e+06
mean    -2.010925e+05
std      6.066505e+05
min     -9.244552e+07
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.000000e-02
dtype: float64


In [13]:
destination_balance_difference = (
    df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
)

print(destination_balance_difference.describe())

count    6.362620e+06
mean     5.556717e+04
std      4.415288e+05
min     -7.588573e+07
25%      0.000000e+00
50%      3.500490e+03
75%      2.935305e+04
max      1.319123e+07
dtype: float64


In [14]:
print("Infinite values:", np.isinf(df.select_dtypes(include=np.number)).sum().sum())

Infinite values: 0


In [15]:
# Create a copy so the original dataframe remains unchanged
cleaned_df = df.copy()

# Remove high-cardinality account identifiers
cleaned_df = cleaned_df.drop(
    columns=["nameOrig", "nameDest"]
)

print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)

Original shape: (6362620, 11)
Cleaned shape: (6362620, 9)


In [16]:
print(cleaned_df.columns.tolist())

['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [17]:
print(cleaned_df.isnull().sum())

step              0
type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [18]:
print("Duplicate rows:", cleaned_df.duplicated().sum())

Duplicate rows: 543


In [19]:
print(cleaned_df["isFraud"].value_counts())

isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [20]:
# output_path = "../data/processed/cleaned_fraud_data.csv"

# cleaned_df.to_csv(output_path, index=False)

# print("Cleaned dataset saved successfully!")
# print(output_path)

In [21]:
print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)
print(cleaned_df.columns.tolist())
print(cleaned_df["isFraud"].value_counts())
print("Duplicate rows:", cleaned_df.duplicated().sum())

Original shape: (6362620, 11)
Cleaned shape: (6362620, 9)
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
isFraud
0    6354407
1       8213
Name: count, dtype: int64
Duplicate rows: 543


In [22]:
duplicate_rows = cleaned_df[cleaned_df.duplicated(keep=False)]

print("Duplicate rows:", len(duplicate_rows))

Duplicate rows: 1081


In [23]:
duplicate_groups = (
    cleaned_df[
        cleaned_df.duplicated(
            subset=[
                "step",
                "type",
                "amount",
                "oldbalanceOrg",
                "newbalanceOrig",
                "oldbalanceDest",
                "newbalanceDest",
                "isFlaggedFraud"
            ],
            keep=False
        )
    ]
)

print(
    duplicate_groups.groupby(
        [
            "step",
            "type",
            "amount",
            "oldbalanceOrg",
            "newbalanceOrig",
            "oldbalanceDest",
            "newbalanceDest",
            "isFlaggedFraud"
        ]
    )["isFraud"].nunique().value_counts()
)

isFraud
1    538
Name: count, dtype: int64


In [24]:
before = len(cleaned_df)

cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)

after = len(cleaned_df)

print("Rows before removing duplicates:", before)
print("Rows after removing duplicates:", after)
print("Duplicates removed:", before - after)

Rows before removing duplicates: 6362620
Rows after removing duplicates: 6362077
Duplicates removed: 543


In [25]:
print("Shape:", cleaned_df.shape)
print("Duplicate rows:", cleaned_df.duplicated().sum())

print("\nTarget distribution:")
print(cleaned_df["isFraud"].value_counts())

print("\nColumns:")
print(cleaned_df.columns.tolist())

Shape: (6362077, 9)
Duplicate rows: 0

Target distribution:
isFraud
0    6353880
1       8197
Name: count, dtype: int64

Columns:
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [26]:
output_path = "../data/processed/cleaned_fraud_data.csv"

cleaned_df.to_csv(output_path, index=False)

print("Cleaned dataset updated successfully!")
print(output_path)

Cleaned dataset updated successfully!
../data/processed/cleaned_fraud_data.csv


In [27]:
# Recreate cleaned dataset from the original raw dataframe
cleaned_df = df.copy()

# Remove high-cardinality account identifiers
cleaned_df = cleaned_df.drop(
    columns=["nameOrig", "nameDest"]
)

print("Cleaned shape:", cleaned_df.shape)

Cleaned shape: (6362620, 9)


In [28]:
print(cleaned_df["isFraud"].value_counts())

isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [29]:
print(cleaned_df.columns.tolist())

['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [30]:
output_path = "../data/processed/cleaned_fraud_data.csv"

cleaned_df.to_csv(output_path, index=False)

print("Final cleaned dataset saved successfully!")
print(output_path)

Final cleaned dataset saved successfully!
../data/processed/cleaned_fraud_data.csv
